[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Guía del tema 03](README.md)

# OpenMP: fork-join y entorno de datos

**Tema:** 03 · **Sesiones:** 11, 12 · **Edición:** 1.0.2026

**Pregunta guía:** ¿Qué variables comparte cada hilo y cuáles deben ser privadas para conservar corrección?


## Cómo usar este notebook

Sigue la secuencia sin saltar directamente al código:

1. Comprueba los prerrequisitos y formula una respuesta inicial a la pregunta guía.
2. Estudia la explicación paso a paso y reconstruye el mapa visual.
3. Predice el resultado del ejemplo resuelto antes de ejecutar su celda.
4. Repite el razonamiento en el ejemplo guiado y contesta las preguntas de comprensión.
5. Solo entonces desarrolla los ejercicios progresivos y contrasta los criterios de aceptación.

La meta no es memorizar una salida: es poder explicar qué se calculó, bajo qué supuestos y con qué evidencia.


## Antes de empezar

**Por qué importa.** OpenMP reduce código ceremonial, pero obliga a clasificar cada variable y a entender dónde comienza y termina el equipo de hilos.

**Prerrequisitos.**

- Memoria compartida, carreras y sincronización.
- Compilación C/C++ con advertencias habilitadas.

**Diagnóstico inicial.** Escribe una respuesta de dos frases a la pregunta guía. Al terminar, vuelve a leerla y señala qué corregiste.


## Resultados de aprendizaje

Al finalizar podrás:

- Explicar fork-join y equipos de hilos.
- Auditar `shared`, `private`, `firstprivate` y reducciones.
- Usar `default(none)` como herramienta de revisión.


## Explicación paso a paso

En esta sección todavía no se busca programar. Primero se construye el modelo mental que permitirá leer el ejemplo y detectar conclusiones inválidas.

### Paso 1: construye la idea

OpenMP crea equipos alrededor de regiones paralelas y sincroniza implícitamente salvo cláusula contraria.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 2: construye la idea

El alcance léxico de C/C++ no basta para deducir el atributo de datos de OpenMP.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Paso 3: construye la idea

Afinidad y schedule afectan localidad, pero no deben cambiar el resultado correcto.

**Pausa de comprensión.** Explica con tus palabras qué supuesto introduce esta idea y qué dato permitiría comprobarlo.

### Vocabulario mínimo

- fork–join — creación y reunión de trabajo paralelo
- entorno de datos — clasificación shared/private/firstprivate
- granularidad — cantidad de trabajo útil por unidad planificada


## Mapa visual

Los diagramas se almacenan en la carpeta compartida [`curso/images/`](../../images/README.md). Úsalos para explicar relaciones y secuencias; no los trates como resultados experimentales.

### Fork Join

![Región serial que crea y reúne trabajadores](../../images/fork-join.svg)

**Cómo leerlo.** La región posterior al join solo puede consumir el resultado cuando todos los trabajadores necesarios terminaron y publicaron sus parciales.

### Distribucion Trabajo

![Iteraciones distribuidas y reducción final](../../images/distribucion-trabajo.svg)

**Cómo leerlo.** Verifica dos propiedades: cada iteración pertenece a un trabajador y la combinación de parciales reproduce la referencia serial.


In [ ]:
from pathlib import Path

def find_repository(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "INDICE_CURSO.md").is_file():
            return candidate
    raise RuntimeError("No se encontró la raíz del repositorio")

ROOT = find_repository(Path.cwd())
TOPIC = "03"
NOTEBOOK = "03_openmp/01_modelo_datos.ipynb"
assert (ROOT / "curso" / "notebooks" / "03_openmp" / "README.md").is_file()
print(f"Repositorio: {ROOT}")
print(f"Notebook: {NOTEBOOK}")


## Ejemplo resuelto: Auditoría de variables

**Situación.** Se documenta el rol de cada dato en una suma paralela.

**Razonamiento antes del código.**

1. Identifica entradas, supuestos y la magnitud que debe producirse.
2. Formula una propiedad esperada; la celda la expresa mediante una aserción.
3. Predice el resultado y después ejecuta. Una salida impresa ayuda a observar, pero la aserción decide si se conserva el invariante.


In [ ]:
variables = {
    "entrada": ("shared", "solo lectura"),
    "n": ("shared", "límite inmutable"),
    "i": ("private", "índice de iteración"),
    "parcial": ("private", "acumulador por hilo"),
    "total": ("reduction", "combinación asociativa definida"),
}
assert variables["i"][0] == "private"
for name, (scope, reason) in variables.items(): print(f"{name:8} {scope:10} {reason}")


### Explicación del resultado

La auditoría convierte `default(none)` en una explicación del diseño y no solo en una exigencia sintáctica.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Ejemplo guiado: Schedule estático

**Situación.** Se visualiza la asignación de iteraciones por bloques contiguos.

**Tu turno antes de ejecutar.** Anota una predicción, identifica la variable que modificarías y explica qué propiedad no debe cambiar. Ejecuta después y compara el resultado con tu predicción.


In [ ]:
n, threads = 19, 4
owner = {}
q, r = divmod(n, threads)
start = 0
for thread in range(threads):
    end = start + q + (thread < r)
    for i in range(start, end): owner[i] = thread
    start = end
assert sorted(owner) == list(range(n))
for thread in range(threads): print(thread, [i for i in owner if owner[i] == thread])


### Lectura razonada

La implementación de `schedule(static)` puede distribuir chunks según la cláusula; se documenta la forma usada en el experimento.

**Qué debes poder explicar.** Relaciona cada valor producido con el modelo conceptual y distingue el cálculo ilustrativo de una medición sobre hardware real.


## Comprueba tu comprensión

1. ¿Qué error ayuda a descubrir `default(none)` antes de ejecutar el programa?
2. ¿Qué aserción o comparación del ejemplo protege la corrección y qué error detectaría?
3. ¿Qué parte es un modelo y qué evidencia adicional exigirías antes de generalizar al hardware real?

Responde primero sin ejecutar código. Luego usa las celdas anteriores para corregir o precisar tu explicación.


## Ejercicios progresivos

### Nivel 1 — reproducir y explicar

Cambia un parámetro del ejemplo resuelto, predice el efecto y explica por qué la aserción debe seguir pasando o debe fallar de manera controlada.

### Nivel 2 — aplicar

1. Compilar `openmp/hello.cc` y observar identificadores de hilo.
2. Agregar `default(none)` a un bucle y clasificar todas las variables.
3. Registrar `OMP_NUM_THREADS`, afinidad y schedule.

### Nivel 3 — producir evidencia

Conserva entrada, comandos, versión del entorno, resultados crudos y una conclusión limitada por los supuestos. Separa siempre corrección, tiempo de kernel y tiempo extremo a extremo cuando corresponda.


## Errores frecuentes

- Asumir que toda variable local es private.
- Escribir salida concurrente y usar su orden como evidencia.
- Cambiar schedule y tamaño a la vez.


## Criterios de aceptación

- Clasificación explícita de datos.
- Salida igual a referencia serial.
- Entorno OpenMP conservado en el informe.


## Síntesis

- La pregunta que debes poder responder es: **¿Qué variables comparte cada hilo y cuáles deben ser privadas para conservar corrección?**
- Los ejemplos convierten el modelo en propiedades comprobables; no sustituyen una medición del sistema objetivo.
- Los ejercicios se consideran terminados cuando la explicación, la corrección y la evidencia satisfacen los criterios de aceptación.


## Referencias y material relacionado

- [Hello OpenMP](../../../openmp/hello.cc)
- [Data sharing](../../../openmp/data_sharing.c)


[← Volver al índice del curso](../../../INDICE_CURSO.md) · [Continuar desde la guía del tema 03](README.md)
